In [1]:
import os
import time
from pathlib import Path
from openai import OpenAI

In [2]:
file_path = "./data/hoanghamobile.csv"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

file_id = ''

vector_store_id = ''

In [3]:
# Validate file exits
if not Path(file_path).exists():
    raise FileNotFoundError(f"File not found: {file_path}")

# Upload file to openai
with open(file_path, "rb") as file_content:
    result = client.files.create(file=file_content, purpose="assistants")
    print(result)
    file_id = result.id   #IMPORTANT: Save this file id for later use

FileObject(id='file-VonYzgUFC9C8i1Ed5sRTPx', bytes=232198, created_at=1766942705, filename='hoanghamobile.csv', object='file', purpose='assistants', status='processed', expires_at=None, status_details=None)


In [4]:
# Create vector store
vector_store_name = "hoanghamobile_store"
vector_store = client.vector_stores.create(name=vector_store_name)
print(vector_store)
vector_store_id = vector_store.id

VectorStore(id='vs_695167f2a55c8191bc88d82441c21cae', created_at=1766942706, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0), last_active_at=1766942706, metadata={}, name='hoanghamobile_store', object='vector_store', status='completed', usage_bytes=0, expires_after=None, expires_at=None, description=None)


In [5]:
# Add file into vector store
result = client.vector_stores.files.create(
    vector_store_id=vector_store_id, file_id=file_id
)
print(result)

VectorStoreFile(id='file-VonYzgUFC9C8i1Ed5sRTPx', created_at=1766942708, last_error=None, object='vector_store.file', status='in_progress', usage_bytes=0, vector_store_id='vs_695167f2a55c8191bc88d82441c21cae', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))


In [6]:
# Get all files from vector store
result = []

# Check if all files are completed
check_interval = 2
max_wait = 60

print("⏳ Checking vector store status...")

start_time = time.time()
while time.time() - start_time < max_wait:
    result = client.vector_stores.files.list(vector_store_id=vector_store_id)

    # Check if all files are completed
    all_completed = all(file.status == "completed" for file in result.data)

    if all_completed:
        print("✅ Vector store is ready! All files processed.")

        # Show current status
    statuses = [f.status for f in result.data]
    print(f"   Status: {statuses}")

    time.sleep(check_interval)

print("⚠️  Timeout waiting for vector store to be ready")
print(result)
all_completed = all(file.status == "completed" for file in result.data)
print(all_completed)

⏳ Checking vector store status...
✅ Vector store is ready! All files processed.
   Status: []
✅ Vector store is ready! All files processed.
   Status: []
   Status: ['in_progress']
   Status: ['in_progress']
   Status: ['in_progress']
✅ Vector store is ready! All files processed.
   Status: ['completed']
✅ Vector store is ready! All files processed.
   Status: ['completed']
✅ Vector store is ready! All files processed.
   Status: ['completed']
✅ Vector store is ready! All files processed.
   Status: ['completed']
✅ Vector store is ready! All files processed.
   Status: ['completed']
✅ Vector store is ready! All files processed.
   Status: ['completed']
✅ Vector store is ready! All files processed.
   Status: ['completed']
✅ Vector store is ready! All files processed.
   Status: ['completed']
✅ Vector store is ready! All files processed.
   Status: ['completed']
✅ Vector store is ready! All files processed.
   Status: ['completed']
✅ Vector store is ready! All files processed.
   Status

In [7]:
vs = client.vector_stores.retrieve(vector_store_id)

vector_store_obj = {
    "id": vs.id,
    "name": vs.name,
    "status": vs.status,
    "file_counts": {
        "total": vs.file_counts.total,
        "completed": vs.file_counts.completed,
        "in_progress": vs.file_counts.in_progress,
        "failed": vs.file_counts.failed,
    },
    "usage_bytes": vs.usage_bytes,
    "created_at": vs.created_at,
}

print(vector_store_obj)

{'id': 'vs_695167f2a55c8191bc88d82441c21cae', 'name': 'hoanghamobile_store', 'status': 'completed', 'file_counts': {'total': 1, 'completed': 1, 'in_progress': 0, 'failed': 0}, 'usage_bytes': 659094, 'created_at': 1766942706}


In [10]:
query = "SIM Nano SIM"

file_search_tool = {
    "type": "file_search",
    "vector_store_ids": [vector_store_id],
    "max_num_results": 5,
}

print(f"🔍 Searching for: {query}")
print(f"📦 Vector Store ID: {vector_store_id}")
print(f"⚙️  Tool config: {file_search_tool}")

response = client.responses.create(
    model="gpt-4o-mini",  # Cheapest model, we only need search results
    input=query,
    tools=[file_search_tool],
    include=["file_search_call.results"],
)

search_results = []

for item in response.output:
    if item.type == "file_search_call" and hasattr(item, "results"):
        if item.results:
            for result in item.results:
                search_results.append(
                    {
                        "content": result.text if hasattr(result, "text") else "",
                        "score": result.score if hasattr(result, "score") else 0.0,
                        "file_id": result.file_id if hasattr(result, "file_id") else "",
                        "filename": result.filename
                        if hasattr(result, "filename")
                        else "",
                    }
                )

for result in search_results:
    print(result)

🔍 Searching for: SIM Nano SIM
📦 Vector Store ID: vs_695167f2a55c8191bc88d82441c21cae
⚙️  Tool config: {'type': 'file_search', 'vector_store_ids': ['vs_695167f2a55c8191bc88d82441c21cae'], 'max_num_results': 5}
{'content': 'Góc mở rộng: 8MP; f/2.2; FOV 112°; ống kính 5P, cố định tiêu cự. Hỗ trợ 2x zoom quang học hybrid và 20x zoom kỹ thuật số. 32MP; f/2.4; FOV 89°; ống kính 5P lens, không hỗ trợ AF hay OIS<br> Kích thước màn hình:\n6.7 inch<br> Hệ điều hành:\nColorOS 13.1 trên nền tảng Android 13<br> Vi xử lý:\nMediaTek Dimensity 7050<br> Bộ nhớ trong:\n256GB<br> RAM:\n8GB<br> Mạng di động:\n2G, 3G, 4G, 5G<br> Số khe SIM:\n2 nano SIM<br> Dung lượng pin:\n5000 mAh<br>","8,990,000 ₫","[\'Xanh Dương\', \'Màu Xám\']"\n122,666baeb79793e149fe739431,https://hoanghamobile.com/dien-thoai-di-dong/tecno-pova-5-8gb-256gb-chinh-hang,điện thoại tecno pova 5 (8+8gb/256gb) - chính hãng,- Ưu đãi trả góp 0% qua Shinhan Finance hoặc Mirae Asset Finance<br>- Giảm 5% không giới hạn khuyến mãi qua Homepaylate

In [9]:
print(search_results)

[{'content': '403 PPI, Màn hình ngoài: 382 x 720, 250 PPI, Camera chính: 50MP IMX890 ƒ/1.8, FOV 84°, tiêu cự 24mm, OIS, Camera Tele: 32MP IMX709 ƒ/2.0, FOV 49°, tiêu cự 47mm, OIS, Camera góc siêu rộng: 48MP IMX581, ƒ/2.2, FOV 114°, tiêu cự 14mm, 32MP ƒ/2.4, FOV 90°, tiêu cự 21 mm<br> Kích thước màn hình:\n6.80 inch, 3.26 inch<br> Hệ điều hành:\nColorOS 13 trên nền tảng Android 13<br> Vi xử lý:\nMediaTek Dimensity 9200<br> Bộ nhớ trong:\n256GB<br> RAM:\n12GB<br> Mạng di động:\n2G, 3G, 4G, 5G<br> Số khe SIM:\n2 nano-SIM<br> Dung lượng pin:\n4300 mAh<br>","19,990,000 ₫","[\'Màu Đen\', \'Màu Vàng\']"\n109,666baeb69793e149fe73941f,https://hoanghamobile.com/dien-thoai-di-dong/tcl-408-4gb-64gb-chinh-hang,điện thoại tcl 408 (4gb/64gb) - chính hãng,- Giảm 5% không giới hạn khuyến mãi qua Homepaylater<br>- Giảm 50% tối đa 700k khi mở thẻ tín dụng Vpbank trên SenID<br>- Giảm 20% tối đa 500k khi mở thẻ tín dụng TPBank EVO<br>- Giảm 1% tối đa 100.000đ  khi thanh toán qua Zalopay<br>,"Công nghệ màn 